# Duplicate CA Cities


#### Step 0: Set-Up
Import the [climakitae](https://github.com/cal-adapt/climakitae) library and other dependencies.

In [2]:
import climakitae as ck

from climakitae.core.data_load import load

import xarray as xr
import pandas as pd
import geopandas as gpd
from shapely.geometry import mapping
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from climakitae.new_core.data_access.boundaries import Boundaries
import intake
import contextily as cx
import pandas as pd
from shapely.geometry import Point, LineString, Polygon
import matplotlib.cm as cm
import matplotlib.colors as mcolors

#### Step 1: Load in city boundaries

In [5]:
catalog = intake.open_catalog('boundaries.yaml')
boundaries = Boundaries(catalog)
# Get all boundary options for UI population
boundary_options = boundaries.boundary_dict()

In [6]:
# Access specific boundary data (loaded lazily)
ca_cities = boundaries._ca_cities
ca_counties= boundaries._ca_counties

#### Step 2: Let's take a look at these duplicates

There 66 duplicate CA city names.

In [ ]:
ca_counties["NAME"].duplicated().sum()

In [ ]:
ca_cities["CDT_NAME_S"].duplicated().sum()

All duplicates contain at least one offshore boundary.

In [7]:
# subsets of duplicate entries and non-duplicate entries
duplicate_cities = ca_cities[ca_cities.duplicated(subset=["CENSUS_GEO"], keep=False)]
non_duplicate_cities = ca_cities[~ca_cities["CENSUS_GEO"].duplicated()]

In [8]:
# now construct a dataframe of city names and whether or not AT LEAST 1 compnent is offshore (ie, "OFFSHORE" not None, so either "bay" or "ocean")
data = []
for name in duplicate_cities["CDT_NAME_S"].unique():
    subset = duplicate_cities[duplicate_cities["CDT_NAME_S"] == name]
    offshore_bool = subset["OFFSHORE"].notna().any()
    data.append({"CDT_NAME_S":name, "offshore_component": offshore_bool, "num_components": len(subset)})
data_df = pd.DataFrame(data)

In [9]:
# 100% of duplicate entries contain at least one offshore element
perc_offshore_dups = sum(data_df["offshore_component"]) / len(duplicate_cities['CENSUS_GEO'].unique()) * 100
perc_offshore_dups

100.0

There are either 2 or 3 components in each subset of duplicates. Only San Fransciso contains 3, as one component is inland, one is ocean-bound, and one is bay-bound.

In [10]:
data_df['num_components'].unique()

array([2, 3])

In [13]:
data_df[data_df["num_components"]==3]

,CDT_NAME_S,offshore_component,num_components
49,San Francisco,True,3


In [14]:
duplicate_cities[duplicate_cities["CDT_NAME_S"] == "San Francisco"]

,CDTFA_COPR,CDTFA_CITY,CDTFA_COUN,CENSUS_PLA,CENSUS_GEO,CENSUS_P_1,GNIS_PLACE,GNIS_ID,CDT_CITY_A,CDT_COUNTY,PRIMARY_DO,CENSUS_POP,CDT_NAME_S,OFFSHORE,AREA_SQMI,GlobalID,geometry
532,38001,San Francisco,San Francisco County,San Francisco,0667000,City,City of San Francisco,2411786,SFSC,SFC,None,0,San Francisco,ocean,130.723377,bcd9363c-7e93-425c-b664-404c6086456e,"MULTIPOLYGON (((-264387.666 -22804.014, -26394..."
370,38001,San Francisco,San Francisco County,San Francisco,0667000,City,City of San Francisco,2411786,SFSC,SFC,None,0,San Francisco,None,47.296328,98194214-ca93-4b62-9e79-082d3ecfd25f,"MULTIPOLYGON (((-212638.26 -14308.404, -212604..."
531,38001,San Francisco,San Francisco County,San Francisco,0667000,City,City of San Francisco,2411786,SFSC,SFC,None,0,San Francisco,bay,54.005066,f40d5a2b-5834-4220-ba9a-44b58888fe9c,"POLYGON ((-208311.681 -12767.978, -208212.197 ..."


#### Step 3: And now to visualize them

Let's map cities with dupicate entries!

In [ ]:
fig, ax = plt.subplots(figsize=(9, 9))

xlim = [-1.385e7, -1.270e7]
ylim = [3.85e6, 5.16e6]
ax.set_ylim(ylim)
ax.set_xlim(xlim)

duplicate_cities = duplicate_cities.to_crs(epsg=3857)

# Build a color map: one color per unique "love" value
categories = duplicate_cities["CENSUS_GEO"].unique()
palette = cm.get_cmap("Set2", len(categories))
color_map = {cat: mcolors.to_hex(palette(i)) for i, cat in enumerate(categories)}

for _, row in duplicate_cities.iterrows():
    geom = row.geometry
    fill_color = color_map[row["CENSUS_GEO"]]

    if geom.geom_type == "Polygon":
        x, y = geom.exterior.xy
        ax.fill(x, y, color=fill_color, alpha=0.5, edgecolor="black")
    elif geom.geom_type == "MultiPolygon":
        for poly in geom.geoms:
            x, y = poly.exterior.xy
            ax.fill(x, y, color=fill_color, alpha=0.5, edgecolor="black")

# Add basemap
cx.add_basemap(ax, source=cx.providers.CartoDB.Positron)

Now take a closer look at Los Angeles. Indeed, one entry is for offshore components of LA, while the other is for inland components.

In [ ]:
# ZOOMED

city_name = "Los Angeles"

# Figure global settings
fig, ax = plt.subplots(figsize=(9, 9))

# Enforce WECC boundary, all plots regardless of variable selection
xlim = [-1.320e7, -1.314e7]  # lon
ylim = [3.975e6, 4.03e6]  # lat
ax.set_ylim(ylim)
ax.set_xlim(xlim)

# Reproject to match Web Mercator (required for contextily basemaps)
city1 = duplicate_cities[duplicate_cities["CDT_NAME_S"] == city_name][0:1]

for part in city1.geometry:
    if part.geom_type == "Polygon":
        x, y = part.exterior.xy
        ax.fill(x, y, color="orange", alpha=0.5, edgecolor="black")
    elif part.geom_type == "MultiPolygon":
        for poly in part.geoms:
            x, y = poly.exterior.xy
            ax.fill(x, y, color="orange", alpha=0.5, edgecolor="black")

# Add basemap
cx.add_basemap(ax, source=cx.providers.CartoDB.Positron)

In [ ]:
city1

In [ ]:
city_name = "Los Angeles"

# Figure global settings
fig, ax = plt.subplots(figsize=(9, 9))

# Enforce WECC boundary, all plots regardless of variable selection
xlim = [-1.325e7, -1.31e7]  # lon
ylim = [3.975e6, 4.16e6]  # lat
ax.set_ylim(ylim)
ax.set_xlim(xlim)

# Reproject to match Web Mercator (required for contextily basemaps)
city2 = duplicate_cities[duplicate_cities["CDT_NAME_S"] == city_name][1:2]

for part in city2.geometry:
    if part.geom_type == "Polygon":
        x, y = part.exterior.xy
        ax.fill(x, y, color="orange", alpha=0.5, edgecolor="black")
    elif part.geom_type == "MultiPolygon":
        for poly in part.geoms:
            x, y = poly.exterior.xy
            ax.fill(x, y, color="orange", alpha=0.5, edgecolor="black")

# Add basemap
cx.add_basemap(ax, source=cx.providers.CartoDB.Positron)

In [ ]:
city2